# Comparação YOLOv11 vs YOLOv26

Este gráfico lê o melhor `mAP@0.5` de validação de cada variante diretamente dos artefatos `results.csv` armazenados nas tags Git publicadas `yolo11-v1.0.0` e `yolo26-v1.0.0`. Nenhuma métrica é inserida manualmente.

Execute a partir da pasta `notebooks/` ou ajuste `PROJECT_ROOT`.

## Gerar gráfico

In [ ]:
import csv
from io import StringIO
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path("..")
VARIANTS = ("n", "s", "m", "l")
METRIC_COLUMN = "metrics/mAP50(B)"

def best_map50_from_tag(tag, family, variant):
    artifact = f"models/{family}{variant}/results.csv"
    result = subprocess.run(
        ["git", "show", f"{tag}:{artifact}"],
        cwd=PROJECT_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    rows = list(csv.DictReader(StringIO(result.stdout)))
    if not rows or METRIC_COLUMN not in rows[0]:
        raise KeyError(f"{METRIC_COLUMN} ausente de {tag}:{artifact}")
    return max(float(row[METRIC_COLUMN]) for row in rows)

yolo11_map50 = [best_map50_from_tag("yolo11-v1.0.0", "yolov11", variant) for variant in VARIANTS]
yolo26_map50 = [best_map50_from_tag("yolo26-v1.0.0", "yolov26", variant) for variant in VARIANTS]

print({"YOLOv11": dict(zip(VARIANTS, yolo11_map50)), "YOLOv26": dict(zip(VARIANTS, yolo26_map50))})

x = np.arange(len(VARIANTS))
width = 0.34
fig, ax = plt.subplots(figsize=(7, 4))
bars11 = ax.bar(x - width / 2, yolo11_map50, width, label="YOLOv11", color="#1f4e79", edgecolor="black", hatch="//")
bars26 = ax.bar(x + width / 2, yolo26_map50, width, label="YOLOv26", color="#f4a261", edgecolor="black", hatch="--")

for bars in (bars11, bars26):
    for bar in bars:
        value = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.002, f"{value:.4f}", ha="center", va="bottom", fontsize=9)

ax.set_title("Comparação de mAP@0.5 - YOLOv11 vs YOLOv26")
ax.set_xlabel("Variante do modelo")
ax.set_ylabel("mAP@0.5")
ax.set_xticks(x, ("nano", "small", "medium", "large"))
ax.set_ylim(0.80, 0.92)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()